In [ ]:
import os
import re
import gc
import time
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

os.environ["OMP_NUM_THREADS"] = "12"
os.environ["MKL_NUM_THREADS"] = "12"
torch.set_num_threads(12)

def clean_text_optimized(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    return " ".join(text.split())

@torch.inference_mode()
def predict_sentiment_scores(texts, tokenizer, model, batch_size=64, max_length=64):
    model.eval()
    scores = []
    
    pbar = tqdm(range(0, len(texts), batch_size), desc="   -> Batch Inference", leave=False)
    
    for i in pbar:
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1)

        if probs.shape[1] == 2:
            batch_scores = (probs[:, 1] - probs[:, 0]).cpu().tolist()
        else:
            batch_scores = (probs[:, -1] - probs[:, 0]).cpu().tolist()

        scores.extend(batch_scores)
    
    return scores

def run_large_scale_bert(
    full_file_path,
    model_name="distilbert-base-uncased-finetuned-sst-2-english",
    chunksize=5000,     
    batch_size=64,      
    max_length=64,      
    temp_save_path="sentiment_scores_raw.csv"
):
    start_time = time.time()
    
    print(f"Initializing Model: {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    if os.path.exists(temp_save_path):
        os.remove(temp_save_path)

    reader = pd.read_csv(
        full_file_path, 
        usecols=["listing_id", "comments"], 
        chunksize=chunksize, 
        compression='gzip'
    )

    print(f"Starting stream processing...")
    
    for chunk_idx, df in enumerate(tqdm(reader, desc="Total Chunks Progress"), start=1):
        
        df = df.dropna(subset=["comments"]).copy()
        df["clean_text"] = df["comments"].map(clean_text_optimized)
        df = df[df["clean_text"].str.len() > 5].copy()

        if df.empty:
            continue

        texts = df["clean_text"].tolist()
        scores = predict_sentiment_scores(
            texts, tokenizer, model,
            batch_size=batch_size,
            max_length=max_length
        )

        df_to_save = pd.DataFrame({
            "listing_id": df["listing_id"],
            "score": scores
        })
        
        df_to_save.to_csv(temp_save_path, mode='a', index=False, header=(not os.path.exists(temp_save_path)))

        # 4. 显式内存管理
        del df, df_to_save, texts, scores
        gc.collect()

    print("\nInference complete. Generating final listing report...")
    raw_results = pd.read_csv(temp_save_path)
    
    final_report = raw_results.groupby("listing_id")["score"].agg(
        review_count="count",
        average_sentiment="mean"
    ).reset_index()

    final_report = final_report[final_report["review_count"] >= 5]
    final_report = final_report.sort_values("average_sentiment", ascending=False)

    end_time = time.time()
    print(f"Total time: {(end_time - start_time)/3600:.2f} hours")
    
    return final_report

if __name__ == "__main__":
    target_file = r"D:\2024-10-18_reviews.csv.gz" 
    
    if os.path.exists(target_file):
        results_df = run_large_scale_bert(target_file)
        results_df.to_csv("airbnb_sentiment_final_report.csv", index=False)
        print("Success! Final results saved.")
    else:
        print("File not found.")